# Bài 3: So sánh Ridge, Decision Tree, SVR, Gradient Boosting với Linear Regression

**Sinh viên:** Nguyễn Đức Duy · **MSV:** 24174800068 · **Lớp:** DHKM18A2HN

Dữ liệu: Diabetes (10 feature). Mỗi thuật toán được chạy với tham số mặc định và sau khi tinh chỉnh bằng `GridSearchCV` (5-fold, chỉ trên tập train).

## 1. Chuẩn bị dữ liệu

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

diabetes = load_diabetes(as_frame=True)
X = diabetes.data.copy()
y = diabetes.target.copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
kf = KFold(5, shuffle=True, random_state=42)
print("Train:", X_train.shape, "| Test:", X_test.shape)

ImportError: DLL load failed while importing arrays: An Application Control policy has blocked this file.

## 2. Hàm đánh giá

In [ ]:
def ev(m):
    """Huấn luyện, đánh giá train/test và R² cross-validation 5-fold."""
    m.fit(X_train, y_train)
    ptr, pte = m.predict(X_train), m.predict(X_test)
    cv = cross_val_score(m, X, y, cv=kf, scoring="r2")
    return dict(MAE_train=mean_absolute_error(y_train, ptr), R2_train=r2_score(y_train, ptr),
                MAE=mean_absolute_error(y_test, pte), RMSE=np.sqrt(mean_squared_error(y_test, pte)),
                R2=r2_score(y_test, pte), CV_R2=cv.mean(), CV_std=cv.std()), pte

## 3. Huấn luyện với tham số mặc định

In [ ]:
default = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(random_state=42),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "SVR": SVR(),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}
rows, preds = [], {}
for n, m in default.items():
    r, p = ev(m)
    rows.append(dict(Model=n, Setting="Mặc định", Params="-", **r))
    preds[(n, "Mặc định")] = p

res_default = pd.DataFrame(rows)
res_default[["Model", "MAE", "RMSE", "R2", "R2_train"]].round(4)

## 4. Tinh chỉnh siêu tham số (GridSearchCV, 5-fold trên tập train)

In [ ]:
grids = {
    "Ridge": (Ridge(random_state=42), {"alpha": [0.001, 0.01, 0.1, 1, 10]}),
    "Decision Tree": (DecisionTreeRegressor(random_state=42),
                      {"max_depth": [2, 3, 4, 5, 6], "min_samples_leaf": [5, 10, 20]}),
    "SVR": (SVR(), {"kernel": ["rbf", "linear"], "C": [1, 10, 100, 300, 1000],
                    "epsilon": [1, 5, 10], "gamma": ["scale", 0.1, 1]}),
    "Gradient Boosting": (GradientBoostingRegressor(random_state=42),
                          {"n_estimators": [50, 100, 200], "learning_rate": [0.03, 0.05, 0.1],
                           "max_depth": [2, 3], "subsample": [0.8, 1.0]}),
}
best = {}
for n, (m, g) in grids.items():
    gs = GridSearchCV(m, g, cv=kf, scoring="neg_mean_squared_error", n_jobs=-1).fit(X_train, y_train)
    best[n] = gs.best_params_
    r, p = ev(gs.best_estimator_)
    rows.append(dict(Model=n, Setting="Tinh chỉnh", Params=str(gs.best_params_), **r))
    preds[(n, "Tinh chỉnh")] = p
    print(f"{n}: {gs.best_params_}")

# Linear Regression không có siêu tham số: bản tinh chỉnh = bản mặc định
lin = [r for r in rows if r["Model"] == "Linear Regression"][0]
rows.append(dict(lin, Setting="Tinh chỉnh", Params="(không có siêu tham số)"))
preds[("Linear Regression", "Tinh chỉnh")] = preds[("Linear Regression", "Mặc định")]
res = pd.DataFrame(rows)

## 5. So sánh kết quả

In [ ]:
order = ["Linear Regression", "Ridge", "Decision Tree", "SVR", "Gradient Boosting"]
tuned = res[res.Setting == "Tinh chỉnh"].set_index("Model").loc[order]
dflt = res[res.Setting == "Mặc định"].set_index("Model").loc[order]
display(tuned[["MAE", "RMSE", "R2", "R2_train", "CV_R2", "CV_std"]].round(4))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
for a, (k, t) in zip(ax, [("MAE", "MAE (thấp hơn tốt hơn)"), ("RMSE", "RMSE (thấp hơn tốt hơn)"), ("R2", "R² (cao hơn tốt hơn)")]):
    x = np.arange(5); w = .38
    a.bar(x - w/2, dflt[k], w, label="Mặc định", color="#b0c4de")
    a.bar(x + w/2, tuned[k], w, label="Tinh chỉnh", color="#1f4e79")
    a.set_xticks(x); a.set_xticklabels(["Linear", "Ridge", "D.Tree", "SVR", "GBR"])
    a.set_title(t); a.grid(alpha=.3, axis="y")
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

fig, a = plt.subplots(figsize=(8, 4)); x = np.arange(5); w = .38
a.bar(x - w/2, tuned.R2_train, w, label="Train", color="#6f9bc8")
a.bar(x + w/2, tuned.R2, w, label="Test", color="#1f4e79")
a.set_xticks(x); a.set_xticklabels(order, fontsize=8); a.set_ylabel("R²")
a.set_title("R² train và test (mô hình tinh chỉnh)"); a.legend(); a.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.show()

## 6. Chẩn đoán mô hình

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(13, 8))
for a, n in zip(ax.ravel(), order):
    p = preds[(n, "Tinh chỉnh")]
    a.scatter(y_test, p, alpha=.65, s=20)
    lo, hi = min(y_test.min(), p.min()), max(y_test.max(), p.max())
    a.plot([lo, hi], [lo, hi], "--", c="r")
    a.set_title(f"{n}\nR²={r2_score(y_test, p):.3f}", fontsize=10)
    a.set_xlabel("Thực tế"); a.set_ylabel("Dự đoán"); a.grid(alpha=.3)
ax[1, 2].axis("off"); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(2, 3, figsize=(13, 8))
for a, n in zip(ax.ravel(), order):
    p = preds[(n, "Tinh chỉnh")]
    a.scatter(p, y_test.values - p, alpha=.65, s=20); a.axhline(0, ls="--", c="r")
    a.set_title(n, fontsize=10); a.set_xlabel("Dự đoán"); a.set_ylabel("Residual"); a.grid(alpha=.3)
ax[1, 2].axis("off"); plt.tight_layout(); plt.show()

# Độ quan trọng feature theo Gradient Boosting
gb = GradientBoostingRegressor(random_state=42, **best["Gradient Boosting"]).fit(X_train, y_train)
imp = pd.Series(gb.feature_importances_, index=X.columns).sort_values()
imp.plot.barh(color="#1f4e79", figsize=(6, 4), title="Độ quan trọng feature (Gradient Boosting)")
plt.tight_layout(); plt.show()

## 7. Nhận xét

- **Linear Regression (mốc):** R² test ≈ 0,453, không overfitting nhưng chỉ ở mức trung bình.
- **Ridge:** `alpha = 1` (mặc định) phạt quá mạnh vì feature có độ lớn nhỏ nên kém hơn Linear (R² 0,419). Tinh chỉnh cho `alpha = 0,001` và kết quả gần như trùng Linear → regularization không mang lại lợi ích rõ trên dữ liệu này.
- **Decision Tree:** mặc định overfit nặng (R² train = 1,00, test ≈ 0,06). Giới hạn độ sâu = 2 thì underfit (R² test ≈ 0,295). Yếu nhất nhóm.
- **SVR:** mặc định underfit (R² ≈ 0,18) do `C = 1` quá nhỏ; tinh chỉnh (`C = 300`, `gamma = 1`, `epsilon = 1`) lên R² ≈ 0,453, ngang Linear.
- **Gradient Boosting:** tốt nhất trên tập test sau tinh chỉnh (R² ≈ 0,470; RMSE ≈ 53,0) nhưng chỉ hơn Linear một chút, nằm trong sai số cross-validation (độ lệch chuẩn ≈ 0,08). Feature quan trọng nhất: `bmi`, `s5`, `bp`.
- **Kết luận:** bốn mô hình tốt nhất có hiệu năng sát nhau (R² 0,453 đến 0,470) → quan hệ chủ yếu tuyến tính và nhiễu cao. Linear/Ridge đơn giản mà hiệu quả gần tương đương; siêu tham số ảnh hưởng rất lớn đến SVR và Decision Tree.